# PWHL intro — sportsdataverse-py

The Professional Women's Hockey League (PWHL) launched its inaugural season in 2024 with six teams: Boston, Minnesota, Montreal, New York, Ottawa, and Toronto. In `sportsdataverse-py` the PWHL is a **loader-only league** — there are no live API wrappers, just `load_pwhl_*` functions that read pre-built parquet releases (schedules, boxscores, play-by-play, scoring/penalty summaries, rosters, and more) and return polars frames.

R companion: [fastRhockey](https://fastRhockey.sportsdataverse.org) (NHL + PWHL). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse.pwhl as pwhl

## Loaders

Every PWHL accessor is a `load_pwhl_*` function. Most take `seasons=[...]` (the inaugural season is `2024`) and all accept `return_as_pandas=True` if you prefer pandas over polars.

In [ ]:
[fn for fn in dir(pwhl) if fn.startswith('load_pwhl_')]

## Schedule

`load_pwhl_schedule(seasons=[2024])` returns one row per game with the result and a set of URL/flag columns pointing at the per-game feeds. Note `home_score`/`away_score` are **strings** — cast before arithmetic.

In [ ]:
schedule = pwhl.load_pwhl_schedule(seasons=[2024])
schedule.shape

In [ ]:
schedule.select([
    'game_id', 'game_date', 'home_team', 'away_team',
    'home_score', 'away_score', 'winner', 'game_type'
]).head()

## Standings

There is no dedicated standings loader, but the schedule's `winner` column makes a regular-season win count a one-liner.

In [ ]:
(schedule
    .filter(pl.col('game_type') == 'regular')
    .group_by('winner')
    .agg(pl.len().alias('wins'))
    .sort('wins', descending=True))

## Rosters

`load_pwhl_rosters(seasons=[2024])` gives one row per player per team, split into skaters and goalies via `player_type`.

In [ ]:
rosters = pwhl.load_pwhl_rosters(seasons=[2024])
rosters.select([
    'team', 'team_abbr', 'player_type', 'first_name', 'last_name',
    'jersey_number', 'position'
]).head()

## Boxscores

Boxscores come in four flavours: `team_box`, `skater_box`, `goalie_box`, and a combined `player_box`. Each is one row per team/player per game.

In [ ]:
team_box = pwhl.load_pwhl_team_box(seasons=[2024])
team_box.select([
    'game_id', 'team', 'team_side', 'goals', 'shots',
    'pp_goals', 'pp_opportunities', 'faceoff_win_pct'
]).head()

In [ ]:
skater_box = pwhl.load_pwhl_skater_box(seasons=[2024])
skater_box.select([
    'game_id', 'first_name', 'last_name', 'position',
    'goals', 'assists', 'points', 'shots', 'plus_minus', 'time_on_ice'
]).head()

In [ ]:
goalie_box = pwhl.load_pwhl_goalie_box(seasons=[2024])
goalie_box.select([
    'game_id', 'first_name', 'last_name',
    'saves', 'shots_against', 'goals_against', 'time_on_ice'
]).head()

## Play-by-play

`load_pwhl_pbp(seasons=[2024])` returns a wide (95-column) event log. The `event` column tags each row as `faceoff`, `shot`, `goal`, or `penalty`, and there are multiple coordinate systems (`x_coord`/`y_coord` plus rink-normalized variants).

In [ ]:
pbp = pwhl.load_pwhl_pbp(seasons=[2024])
pbp.shape

In [ ]:
pbp.select([
    'game_id', 'period_of_game', 'clock', 'event',
    'player_name_first', 'player_name_last', 'x_coord', 'y_coord'
]).head()

In [ ]:
(pbp
    .group_by('event')
    .agg(pl.len().alias('events'))
    .sort('events', descending=True))

## Scoring & penalty summaries

`load_pwhl_scoring_summary` is a tidy goal log (scorer + up to two assists, plus situation flags like power play / short handed). `load_pwhl_three_stars` carries the post-game three-star selections.

In [ ]:
scoring = pwhl.load_pwhl_scoring_summary(seasons=[2024])
scoring.select([
    'game_id', 'period', 'time', 'team_abbr',
    'scorer_first', 'scorer_last', 'is_power_play', 'is_game_winning'
]).head()

## Pipeline example: season scoring leaders

Aggregate the skater boxscore across all games to build a points leaderboard — the inaugural-season top of the table.

In [ ]:
(skater_box
    .group_by(['player_id', 'first_name', 'last_name'])
    .agg(
        pl.col('goals').sum().alias('goals'),
        pl.col('assists').sum().alias('assists'),
        pl.col('points').sum().alias('points'),
    )
    .sort('points', descending=True)
    .select(['first_name', 'last_name', 'goals', 'assists', 'points'])
    .head(10))

## Live API wrappers

In addition to the loader functions above, sportsdataverse-py includes **live HockeyTech wrappers** that
pull directly from the PWHL stats API. These require network access and are skipped in offline CI.
R companion: [`fastRhockey::pwhl_schedule()`](https://fastRhockey.sportsdataverse.org/reference/pwhl_schedule.html).

In [ ]:
# Live schedule for the 2025 season
live_sched = pwhl.pwhl_schedule(season=2025)
live_sched.head()

### Live play-by-play

`pwhl_pbp(game_id)` fetches and enriches a single game: shot coordinates are normalised
to one end, `shot_distance` is computed from the net, and on-ice skater columns
(`on_ice_home_*` / `on_ice_away_*`) are filled from the shift data.

In [ ]:
# Live PBP for a single game -- a few enriched columns
pbp_live = pwhl.pwhl_pbp(game_id=42)
pbp_live.select([
    'game_id', 'period', 'time_elapsed', 'event_type',
    'x_coord', 'y_coord', 'shot_distance', 'on_ice_home_1', 'on_ice_away_1'
]).head()

## Analytics: shifts, TOI, and Corsi

Three analytics helpers are built on top of the live wrappers:

| Function | Description |
|---|---|
| `pwhl_game_shifts(game_id)` | Raw shift stints -- one row per player stint |
| `pwhl_player_toi(game_id)` | Summed time-on-ice per player |
| `pwhl_game_corsi(game_id)` | On-ice shot-attempt counts + Corsi/60 |

In [ ]:
# Shift stints for a single game
shifts = pwhl.pwhl_game_shifts(game_id=42)
shifts.select(['game_id', 'player_id', 'player_name', 'team',
               'period', 'shift_start', 'shift_end', 'duration_s']).head()

In [ ]:
# Per-player time-on-ice totals
toi = pwhl.pwhl_player_toi(game_id=42)
toi.select(['player_id', 'player_name', 'team', 'toi_seconds']).sort('toi_seconds', descending=True).head()

In [ ]:
# On-ice Corsi and Fenwick for a single game
corsi = pwhl.pwhl_game_corsi(game_id=42)
corsi.select(['player_id', 'corsi_for', 'corsi_against', 'corsi_net', 'corsi_for_per60']).sort('corsi_for_per60', descending=True).head()

## Junior leagues (OHL, WHL, AHL, QMJHL)

The same HockeyTech family of wrappers is available for four junior / minor leagues
under their own sub-modules. Each exposes the same function set (`*_schedule`,
`*_pbp`, `*_standings`, `*_teams`, `*_game_shifts`, `*_game_corsi`, ...)
using the league prefix. There are no fastRhockey equivalents yet -- Part B of
the sportsdataverse roadmap adds them.

In [ ]:
# OHL example -- same API surface, different league
import sportsdataverse.ohl as ohl
ohl.ohl_schedule().head()

## Cross-references

- R companion: [fastRhockey](https://fastRhockey.sportsdataverse.org) (NHL + PWHL)
- Data source: PWHL HockeyTech feeds, packaged as parquet in the sportsdataverse-data releases
- Plotting: matplotlib, plotnine (rink plots from the `x_coord`/`y_coord` PBP columns)

## Where to go next

- API docs: `docs/docs/pwhl/index.md`
- For the men's game and the modern NHL APIs, see `07_nhl_intro.ipynb`.